In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, update_display, Markdown
import json


load_dotenv(override=True)

True

In [2]:
import gradio as gr

In [3]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if not openai_api_key:
    print("No API Key found for OpenAI")
if not openai_api_key.startswith('sk-proj'):
    print("OpenAI API Key is invlaid")

else:
    print("Valid OpenAI API Key found")

if not google_api_key:
    print("No API Key found for Google")
if not google_api_key.startswith('AI'):
    print("Google API Key is invlaid")

else:
    print("Valid Google API Key found")

Valid OpenAI API Key found
Valid Google API Key found


In [4]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI()
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
# we use chat completions api but can also use responses api (more expensive as it uses managed tools & is OpenAI only)
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    messages = [
        {"role": "system", "content":system_message},
        {"role": "user", "content": prompt}]
    response = openai.chat.completions.create(
        model='gpt-4o-mini',
        messages=messages
    )
    return response.choices[0].message.content

In [ ]:
# only trained up to a date and doesnt use tools -> response = unhelpful
message_gpt('What is the weather in london like now?')

'I\'m unable to provide real-time information such as current weather. However, you can easily check the latest weather conditions in London using a weather website, app, or by searching "London weather" in a search engine for the most up-to-date information.'

## Building a Gradio interface

In [13]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

shout('hello')

Shout has been called with input hello


'HELLO'

In [ ]:
from langchain_core import outputs


gr.Interface(fn=shout, #fn = function (doesnt call it btu gradio can call in the future) a.k.a callback
             inputs = "textbox",
             outputs = "textbox",
             flagging_mode='never'
             ).launch() #add inbrowser=True to launch broswer instead of in notebook

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Shout has been called with input Hi1
Shout has been called with input Hello my name is Alexander



In [ ]:
# Adding share=True means that it can be accessed publically
# A more permanent hosting is available using a platform called Spaces from HuggingFace, which we will touch on next week
# NOTE: Some Anti-virus software and Corporate Firewalls might not like you using share=True. 
# If you're at work on on a work network, I suggest skip this test.
#a.k.a HTTP Tunneling

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(share=True)

In [16]:
# adding authentication

gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch(auth=("Alexander","!Mara1234"))

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
#forcing different modes
force_dark_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
"""
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never", js=force_dark_mode).launch()

In [17]:
# more functionality
message_input = gr.Textbox(label="Your Message", info="enter a message to be shouted", lines=7)
message_output = gr.Textbox(label='GPT Response', lines=8)

view = gr.Interface(
    fn =shout,
    inputs=[message_input],
    outputs=[message_output],
    flagging_mode='never'
)

view.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [ ]:
#combine with gpt_message
message_input = gr.Textbox(label="Enter your message", lines=7 )
message_output = gr.Textbox(label='GPT', info = 'OpenAI GPT-4o-mini', lines=8)

view = gr.Interface(
    fn= message_gpt,
    title="OpenAI GPT",
    inputs=[message_input],
    outputs=[message_output],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [22]:
#streaming

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content":system_message},
        {"role":"user", "content":prompt}
    ]
    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        stream=True
    )
    result=""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [25]:
#add to gradio
message_input = gr.Textbox(label="Enter your message", lines=7 )
message_output = gr.Textbox(label='OpenAI GPT', info = 'Model: GPT-4o-mini', lines=8)

view = gr.Interface(
    fn= stream_gpt,
    title="AI Assistant",
    examples=[
        "Explain how to do linear programming for manufacturing production optimisation",
        "Explain Transformer Architecure in laymans terms"
    ],
    inputs=[message_input],
    outputs=[message_output],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## Multimodal

In [27]:
#gpt 
def stream_gpt(prompt):
    messages = [
        {"role": "system", "content":system_message},
        {"role":"user", "content":prompt}
    ]
    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        stream=True
    )
    result=""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

def stream_gemini(prompt):
    messages =[
        {"role":"system", "content": system_message},
        {'role':'user', "content": prompt},
    ]
    stream = gemini.chat.completions.create(
    model = 'gemini-2.5-flash',
    messages=messages,
    stream = True
    )

    result=""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

def stream_model(prompt, model):
    if model =="GPT":
        result = stream_gpt(prompt)
    elif model == "Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model. Please sselect GPT or Gemini")
    yield from result


In [30]:
message_input = gr.Textbox(label = "Enter your message", lines = 7)
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select Model", value="GPT")
message_output = gr.Markdown(label = "AI Response")

view = gr.Interface(
    fn = stream_model,
    title="Multimodal Assistant",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples= [
        ["Explain how to do linear programming for manufacturing production optimisation"],
        ["Explain Transformer Architecure in laymans terms"]
    ],
    flagging_mode='never'
)

view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## Gradio + Brochure Generator

In [ ]:
#tools from scraper.py

from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# scrapers

def fetch_web_contents(url):
    """ 
    Return the title and the contents of the website of the given url.
    Truncate to 2,000 characters as a sensible list
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text=""
    return (title + "\n\n" +text)[:2000]

def fetch_website_links(url): 
    """  
    Return the links on the website of the given url
    """
    response = requests.get(url, headers=headers) #http get request
    soup = BeautifulSoup(response.content, "html.parser") #parses website content (HTML) into a structured object called soup (can search through this)
    links = [link.get("href") for link in soup.find_all("a")] #find <a> anchor tags for links on the website and stores in list
    return [link for link in links if link] #href = hypertext reference

In [ ]:
#turn aboive into a class

class WebsiteLinkFetcher:
    def __init__(self, headers=headers):
        self.headers = headers

    def get_soup(self, url): #function to simplify parsing web content
        response = requests.get(url, headers = headers)
        return BeautifulSoup(response.content, "html.parser")
    
    def fetch_web_content(self, url):
        soup = self.get_soup(url)
        title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text=""
        return (title + "\n\n" +text)[:2000]
    
    def fetch_website_links(self,url):
        soup = self.get_soup(url)
        links = [link.get("href") for link in soup.find_all("a")]
        return [link for link in links if link]

    link_system_prompt = """ 
You are provided with a list of links on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, Career or products page.
You must respond in JSON Format as in the example below:
{
    "links": [
        {"type": "about page", "url": "https://company.com/about"},
        {"type": "products page", "url": "https://company.com/products"},
    ]
}
"""

    def get_links_user_prompt(self, url):
        prompt =f"""  
Here is a list of links on the website {url}. Please decide which links are relevant links for the website brochure
about the company. Respond with the full https in JSON format. Do not include links about service, privacy or email links.

links (some might be relative links):
"""
        links = self.fetch_website_links(url)
        prompt += f'\n {links}'
        return prompt
    
    def select_relevant_links(self, url):
        response = openai.chat.completions.create(
            model='gpt-4o-nano',
            messages = [
                {"role": "system", "content": self.link_system_prompt}
                {"role": "user", "content": self.get_links_user_prompt(url)}
            ],
            response_format= {"typr": "json_object"}
        )

        result = response.choices[0].message.content
        links = json.loads(links)
        return links
    
    def fetch_page_and_relevant_links(self, url):
        content = self.fetch_web_content(url)
        relevant_links = self.select_relevant_links(url)
        result = f"## Landing Page:\n\n {content}\n\n Relevant Links:\n"
        for link in relevant_links:
            result+= f"\n Link: {link['type']}\n"
            result+= self.fetch_web_content(link["url"])
        return url


In [ ]:
system_prompt = """ 
You are provided with a list of links on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, Career or products page.
You must respond in JSON Format as in the example below:
{
    "links": [
        {"type": "about page", "url": "https://company.com/about"},
        {"type": "products page", "url": "https://company.com/products"},
    ]
}
"""

def get_user_prompt()

In [ ]:
# our brouchure generator

def stream(company_name, url):
    